# Dyson Equalizer：下側外れ値の乱数実験

## 目的

少数の低分散方向によってMP下端より小さい固有値を作り、既存の `dyson_equalizer_algorithm1` がそれをbulkへ戻すか検証する。同時に、DEの仮定を満たすモデルと意図的に破るモデルを比較する。

評価するモデルは次の4つ。

| model | 構成 | DEの仮定 |
|---|---|---|
| `A_separable_lower_spike` | 少数の行だけ低分散、他はほぼ分散1 | 満たす |
| `B_separable_plus_signal` | A + 非局在低ランク信号 | 満たす |
| `C_correlated_columns` | A + AR(1)列相関 | 独立性を破る |
| `D_rank_deficient` | 行空間を低ランク射影 | 独立性・非退化性を破る |

スペクトルは `lambda = singular_value**2 / n` で統一する。DE前はBEMA推定分散、DE後は分散1のMP分布と比較する。

### 仮説

- A：上側ノイズ外れ値をほとんど増やさず、下側外れ値だけがDE後にbulkへ戻る。
- B：上側の低ランク信号は残り、下側ノイズ外れ値はbulkへ戻る。
- C：行列要素の相関は対角スケーリングでは除けず、DE後もMP不適合が残る。
- D：ランク欠損によるゼロ特異値は、対角スケーリングでは回復しない。

仮定を破れば必ず失敗するという主張ではない。C・Dは、DEで除去できない構造を明示的に入れたnegative controlである。


## 1. セットアップ

CPUだけで実行できる。VS CodeからColabを使う場合、RMT_utilsが `/content/RMT_utils` にないときだけcloneする。


In [ ]:
import json
import subprocess
import sys
from pathlib import Path


def find_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path("/content/RMT_utils")]
    for candidate in candidates:
        if (candidate / "funcs1.py").exists():
            return candidate.resolve()
    target = Path("/content/RMT_utils")
    if Path("/content").exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/ryy1210/RMT_utils.git", str(target),
        ])
        return target
    raise FileNotFoundError("RMT_utilsのルートで開くか、Colabで実行してください。")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("RMT_utils:", REPO_ROOT)


In [ ]:
import hashlib
import math
import warnings
from dataclasses import asdict, dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from funcs1 import (
    _mp_ks,
    bema_algorithm1_from_eigenvalues,
    dyson_equalizer_algorithm1,
    tw1_quantile,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


## 2. 設定

主実験は長方行列 `p < n` とする。正方行列ではMP下端が0になるため、「下端より小さい」という診断には向かない。

`tau < 1 - sqrt(gamma)` が下側population spikeの理論的な分離条件。既定値では `gamma=0.5`, `tau=0.1` なので明確に分離する。


In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    p: int = 256
    n: int = 512
    lower_spike_count: int = 5
    tau: float = 0.10
    signal_rank: int = 3
    signal_strengths: tuple = (3.2, 2.7, 2.3)  # sqrt(n)倍される
    ar1_rho: float = 0.80
    rank_deficiency: int = 5
    bema_alpha: float = 0.20
    tw_beta: float = 0.10
    near_hard_edge_gap: int = 10
    normalize_mean_variance: bool = True
    repeats: int = 30
    base_seed: int = 20260928


CONFIG = ExperimentConfig()
MODEL_NAMES = [
    "A_separable_lower_spike",
    "B_separable_plus_signal",
    "C_correlated_columns",
    "D_rank_deficient",
]

RUN_SINGLE_EXAMPLES = False
RUN_MONTE_CARLO = False
RUN_TAU_SWEEP = False
TAU_GRID = [0.05, 0.10, 0.20, 0.30, 0.40]

OUTPUT_DIR = REPO_ROOT / "outputs" / "de_synthetic_lower_edge"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gamma = CONFIG.p / CONFIG.n
critical_tau = 1 - np.sqrt(gamma)
mp_lower = (1 - np.sqrt(gamma)) ** 2
mp_upper = (1 + np.sqrt(gamma)) ** 2
print(f"gamma={gamma:.3f}, lower-spike threshold={critical_tau:.4f}")
print(f"MP support for variance 1: [{mp_lower:.4f}, {mp_upper:.4f}]")
print("tau is theoretically separated:", CONFIG.tau < critical_tau)


## 3. 乱数モデル

モデルAの分散プロファイルは

\[
a_i=\begin{cases}\tau&i\in I_{\rm low}\\1&i\notin I_{\rm low}\end{cases},
\qquad E=D_a^{1/2}Z.
\]

平均分散を厳密に1へ合わせる場合も、通常行へ掛かる補正は `1 + O(k/p)` なので上側spikeは作らない。


In [ ]:
def validate_config(config):
    if not (0 < config.p <= config.n):
        raise ValueError("Require 0 < p <= n")
    if not (0 < config.tau < 1):
        raise ValueError("tau must be in (0, 1)")
    if not (0 < config.lower_spike_count < config.p):
        raise ValueError("lower_spike_count must be between 1 and p-1")
    if not (0 <= config.ar1_rho < 1):
        raise ValueError("ar1_rho must be in [0, 1)")
    if not (0 < config.rank_deficiency < config.p):
        raise ValueError("rank_deficiency must be between 1 and p-1")
    if len(config.signal_strengths) != config.signal_rank:
        raise ValueError("signal_strengths length must equal signal_rank")


def lower_spike_profile(p, k, tau, rng, normalize_mean=True):
    indices = np.sort(rng.choice(p, size=k, replace=False))
    profile = np.ones(p, dtype=float)
    profile[indices] = tau
    if normalize_mean:
        profile /= profile.mean()
    return profile, indices


def ar1_cholesky(n, rho):
    index = np.arange(n)
    covariance = rho ** np.abs(index[:, None] - index[None, :])
    return np.linalg.cholesky(covariance)


def delocalized_signal(p, n, rank, strengths, rng):
    left, _ = np.linalg.qr(rng.normal(size=(p, rank)))
    right, _ = np.linalg.qr(rng.normal(size=(n, rank)))
    singular_values = np.sqrt(n) * np.asarray(strengths, dtype=float)
    signal = (left * singular_values) @ right.T
    return signal, left, right, singular_values


def generate_matrix(model_name, config, seed):
    validate_config(config)
    rng = np.random.default_rng(seed)
    z = rng.normal(size=(config.p, config.n))
    a, low_rows = lower_spike_profile(
        config.p, config.lower_spike_count, config.tau, rng,
        normalize_mean=config.normalize_mean_variance,
    )
    b = np.ones(config.n, dtype=float)
    signal = np.zeros((config.p, config.n), dtype=float)
    metadata = {
        "model": model_name,
        "seed": seed,
        "assumption_status": "satisfied",
        "low_rows": low_rows.tolist(),
        "true_a": a,
        "true_b": b,
    }

    if model_name == "A_separable_lower_spike":
        noise = np.sqrt(a)[:, None] * z
    elif model_name == "B_separable_plus_signal":
        noise = np.sqrt(a)[:, None] * z
        signal, left, right, singular_values = delocalized_signal(
            config.p, config.n, config.signal_rank, config.signal_strengths, rng,
        )
        metadata.update({
            "signal_singular_values": singular_values,
            "max_left_entry": float(np.abs(left).max()),
            "max_right_entry": float(np.abs(right).max()),
        })
    elif model_name == "C_correlated_columns":
        # 各列間にAR(1)相関を入れる。周辺分散は保つが、要素独立性を破る。
        chol = ar1_cholesky(config.n, config.ar1_rho)
        noise = (np.sqrt(a)[:, None] * z) @ chol.T
        metadata["assumption_status"] = "violated: correlated entries"
    elif model_name == "D_rank_deficient":
        # ランダムな行方向を完全に落とす。対角スケーリングではrankを回復できない。
        q, _ = np.linalg.qr(rng.normal(size=(config.p, config.rank_deficiency)))
        projection = np.eye(config.p) - q @ q.T
        noise = projection @ z
        metadata.update({
            "assumption_status": "violated: correlated and rank-deficient",
            "true_a": None,
            "rank_upper_bound": config.p - config.rank_deficiency,
        })
    else:
        raise KeyError(model_name)

    observed = signal + noise
    if observed.shape != (config.p, config.n) or not np.isfinite(observed).all():
        raise FloatingPointError("Generated matrix is invalid")
    metadata["signal_frobenius_ratio"] = float(
        np.linalg.norm(signal, "fro") / max(np.linalg.norm(noise, "fro"), np.finfo(float).tiny)
    )
    return observed, signal, noise, metadata


## 4. ESDと両端bulkKS

DE前はBEMAが返す分散と上端閾値を使う。DE後は分散を1に固定し、同じ `tw_beta` による上端・下端補正を使う。下端はsoft edgeの場合だけ評価し、正方またはnear-hard-edgeでは0とする。


In [ ]:
def eigenvalues_from_matrix(matrix):
    p, n = matrix.shape
    if p > n:
        matrix = matrix.T
        p, n = matrix.shape
    singular_values = np.linalg.svd(matrix, compute_uv=False)
    return np.sort(singular_values**2 / n), p, n


def mp_edges(gamma, sigma2):
    return (
        sigma2 * (1 - np.sqrt(gamma)) ** 2,
        sigma2 * (1 + np.sqrt(gamma)) ** 2,
    )


def tw_thresholds(p, n, sigma2, beta, near_hard_edge_gap):
    gamma = p / n
    lower_edge, upper_edge = mp_edges(gamma, sigma2)
    coefficient = float(tw1_quantile(beta=beta))
    upper_scale = sigma2 * n ** (-2 / 3) * gamma ** (-1 / 6) * (1 + np.sqrt(gamma)) ** (4 / 3)
    upper = upper_edge + coefficient * upper_scale
    if p == n or n - p <= near_hard_edge_gap:
        lower, lower_scale, status = 0.0, 0.0, "hard_or_near_hard_edge"
    else:
        lower_scale = sigma2 * n ** (-2 / 3) * gamma ** (-1 / 6) * (1 - np.sqrt(gamma)) ** (4 / 3)
        lower = max(0.0, lower_edge - coefficient * lower_scale)
        status = "soft_edge"
    return {
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mp_lower": lower_edge,
        "mp_upper": upper_edge,
        "tw_scale_lower": lower_scale,
        "tw_scale_upper": upper_scale,
        "tw_coefficient": coefficient,
        "lower_edge_status": status,
    }


def two_sided_diagnostics(eigenvalues, p, n, sigma2, thresholds):
    values = np.sort(np.asarray(eigenvalues, dtype=float))
    low = values < thresholds["lower_threshold"]
    high = values > thresholds["upper_threshold"]
    kept = ~(low | high)
    bulk = values[kept]
    return {
        "fullKS": _mp_ks(values, p / n, sigma2),
        "two_sided_bulkKS": _mp_ks(bulk, p / n, sigma2) if len(bulk) else np.nan,
        "n_total": len(values),
        "n_low": int(low.sum()),
        "n_high": int(high.sum()),
        "n_bulk": int(kept.sum()),
        "low_ratio": float(low.mean()),
        "high_ratio": float(high.mean()),
        "bulk_ratio": float(kept.mean()),
        **thresholds,
    }


def centered_log_error(estimate, truth):
    estimate = np.asarray(estimate, dtype=float)
    truth = np.asarray(truth, dtype=float)
    if len(estimate) != len(truth) or np.any(estimate <= 0) or np.any(truth <= 0):
        return np.nan, np.nan
    log_estimate = np.log(estimate) - np.log(estimate).mean()
    log_truth = np.log(truth) - np.log(truth).mean()
    corr = np.corrcoef(log_estimate, log_truth)[0, 1] if np.std(log_truth) > 0 else np.nan
    rmse = np.sqrt(np.mean((log_estimate - log_truth) ** 2))
    return float(corr), float(rmse)


def analyze_one(model_name, config, seed):
    observed, signal, noise, metadata = generate_matrix(model_name, config, seed)
    pre_eigenvalues, p, n = eigenvalues_from_matrix(observed)
    bema = bema_algorithm1_from_eigenvalues(
        pre_eigenvalues, p=p, n=n, alpha=config.bema_alpha, beta=config.tw_beta,
    )
    pre_thresholds = tw_thresholds(
        p, n, bema["sigma2_hat"], config.tw_beta, config.near_hard_edge_gap,
    )
    # BEMA実装と同じ上端値を使用し、下端だけ対応するTW尺度から構成する。
    pre_thresholds["upper_threshold"] = float(bema["threshold"])
    pre = two_sided_diagnostics(pre_eigenvalues, p, n, bema["sigma2_hat"], pre_thresholds)

    equalized, x_hat, y_hat = dyson_equalizer_algorithm1(observed, full_matrices=False)
    post_eigenvalues, _, _ = eigenvalues_from_matrix(equalized)
    post_thresholds = tw_thresholds(p, n, 1.0, config.tw_beta, config.near_hard_edge_gap)
    post = two_sided_diagnostics(post_eigenvalues, p, n, 1.0, post_thresholds)

    a_corr = a_rmse = b_corr = b_rmse = np.nan
    if metadata["true_a"] is not None:
        a_corr, a_rmse = centered_log_error(x_hat, metadata["true_a"])
        b_corr, b_rmse = centered_log_error(y_hat, metadata["true_b"])

    theoretical_outlier = np.nan
    gamma = p / n
    # 下側spikeの理論式は独立なseparable modelでだけ使用する。
    if model_name in {"A_separable_lower_spike", "B_separable_plus_signal"}:
        effective_tau = float(np.min(metadata["true_a"]))
        if effective_tau < 1 - np.sqrt(gamma):
            theoretical_outlier = effective_tau * (1 + gamma / (effective_tau - 1))

    row = {
        "model": model_name,
        "seed": seed,
        "assumption_status": metadata["assumption_status"],
        "p": p,
        "n": n,
        "gamma": gamma,
        "tau": config.tau,
        "lower_spike_count": config.lower_spike_count,
        "pre_sigma2_bema": float(bema["sigma2_hat"]),
        "theoretical_lower_outlier": theoretical_outlier,
        "x_log_corr": a_corr,
        "x_log_rmse": a_rmse,
        "y_log_corr": b_corr,
        "y_log_rmse": b_rmse,
        "signal_frobenius_ratio": metadata["signal_frobenius_ratio"],
    }
    row.update({f"pre_{key}": value for key, value in pre.items()})
    row.update({f"post_{key}": value for key, value in post.items()})
    row["delta_low_ratio_post_minus_pre"] = row["post_low_ratio"] - row["pre_low_ratio"]
    row["delta_fullKS_post_minus_pre"] = row["post_fullKS"] - row["pre_fullKS"]
    row["delta_bulkKS_post_minus_pre"] = row["post_two_sided_bulkKS"] - row["pre_two_sided_bulkKS"]
    arrays = {
        "observed": observed,
        "equalized": equalized,
        "pre_eigenvalues": pre_eigenvalues,
        "post_eigenvalues": post_eigenvalues,
        "x_hat": x_hat,
        "y_hat": y_hat,
        "metadata": metadata,
    }
    return row, arrays


## 5. 1標本の可視化

各モデルについてDE前後を並べる。赤の破線は上下の判定閾値、黒の点線は理論MP端点。ヒストグラムは密度表示で、対応するMP密度を重ねる。


In [ ]:
def mp_density(grid, gamma, sigma2):
    lower, upper = mp_edges(gamma, sigma2)
    density = np.zeros_like(grid, dtype=float)
    valid = (grid > lower) & (grid < upper)
    density[valid] = np.sqrt((upper - grid[valid]) * (grid[valid] - lower)) / (
        2 * np.pi * gamma * sigma2 * grid[valid]
    )
    return density


def plot_one_result(row, arrays, bins=70):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), layout="constrained")
    conditions = [
        ("pre", arrays["pre_eigenvalues"], row["pre_sigma2_bema"], "Before DE / BEMA variance"),
        ("post", arrays["post_eigenvalues"], 1.0, "After DE / variance 1"),
    ]
    for axis, (prefix, values, sigma2, title) in zip(axes, conditions):
        upper_plot = max(np.quantile(values, 0.995), row[f"{prefix}_mp_upper"] * 1.1)
        grid = np.linspace(max(1e-8, values.min()), upper_plot, 800)
        axis.hist(values, bins=bins, range=(0, upper_plot), density=True, alpha=0.42, color="#4C78A8")
        axis.plot(grid, mp_density(grid, row["gamma"], sigma2), color="#E45756", lw=2, label="MP density")
        axis.axvline(row[f"{prefix}_mp_lower"], color="black", ls=":", lw=1.2)
        axis.axvline(row[f"{prefix}_mp_upper"], color="black", ls=":", lw=1.2)
        axis.axvline(row[f"{prefix}_lower_threshold"], color="#D62728", ls="--", lw=1.2)
        axis.axvline(row[f"{prefix}_upper_threshold"], color="#D62728", ls="--", lw=1.2)
        axis.set(
            title=title,
            xlabel=r"$\lambda=s^2/n$",
            ylabel="density",
            xlim=(0, upper_plot),
        )
        stats = (
            f"fullKS={row[f'{prefix}_fullKS']:.3f}\n"
            f"bulkKS={row[f'{prefix}_two_sided_bulkKS']:.3f}\n"
            f"low={row[f'{prefix}_n_low']}, high={row[f'{prefix}_n_high']}"
        )
        axis.text(0.98, 0.96, stats, transform=axis.transAxes, ha="right", va="top",
                  bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85})
        axis.legend(loc="upper left")
    fig.suptitle(f"{row['model']} — seed={row['seed']} — {row['assumption_status']}")
    return fig


if RUN_SINGLE_EXAMPLES:
    single_rows = []
    for offset, model_name in enumerate(MODEL_NAMES):
        row, arrays = analyze_one(model_name, CONFIG, CONFIG.base_seed + offset)
        single_rows.append(row)
        figure = plot_one_result(row, arrays)
        figure.savefig(OUTPUT_DIR / f"single_{model_name}.png", dpi=170)
        plt.show()
    single_results = pd.DataFrame(single_rows)
    single_results.to_csv(OUTPUT_DIR / "single_results.csv", index=False)
    display(single_results[[
        "model", "assumption_status", "pre_n_low", "post_n_low", "pre_n_high", "post_n_high",
        "pre_fullKS", "post_fullKS", "pre_two_sided_bulkKS", "post_two_sided_bulkKS",
        "x_log_corr", "x_log_rmse",
    ]])
else:
    print("1標本の可視化はRUN_SINGLE_EXAMPLES=Trueで実行します。")


## 6. Monte Carlo比較

1回の偶然ではなく、各モデルを複数seedで比較する。主要指標は `n_low`, `n_high`, `fullKS`, `two_sided_bulkKS` のDE前後差。


In [ ]:
def run_monte_carlo(config, model_names=MODEL_NAMES):
    records = []
    total = len(model_names) * config.repeats
    completed = 0
    for model_index, model_name in enumerate(model_names):
        for repeat in range(config.repeats):
            seed = config.base_seed + model_index * 100_000 + repeat
            row, _ = analyze_one(model_name, config, seed)
            records.append(row)
            completed += 1
            if completed % 10 == 0 or completed == total:
                print(f"[{completed}/{total}] completed")
    return pd.DataFrame(records)


def summarize_monte_carlo(results):
    metrics = [
        "pre_n_low", "post_n_low", "pre_n_high", "post_n_high",
        "pre_fullKS", "post_fullKS",
        "pre_two_sided_bulkKS", "post_two_sided_bulkKS",
        "delta_low_ratio_post_minus_pre", "delta_fullKS_post_minus_pre",
    ]
    def q10(values):
        return values.quantile(0.10)

    def q90(values):
        return values.quantile(0.90)

    summary = results.groupby(["model", "assumption_status"])[metrics].agg(["median", q10, q90])
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    return summary.reset_index()


def plot_monte_carlo(results):
    long = results.melt(
        id_vars=["model", "seed", "assumption_status"],
        value_vars=["pre_n_low", "post_n_low"],
        var_name="condition", value_name="lower_outlier_count",
    )
    long["condition"] = long["condition"].map({"pre_n_low": "Before DE", "post_n_low": "After DE"})
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), layout="constrained")
    sns.boxplot(data=long, x="model", y="lower_outlier_count", hue="condition", ax=axes[0])
    axes[0].tick_params(axis="x", rotation=25)
    axes[0].set_title("Lower-edge outliers before and after DE")
    delta = results.copy()
    delta["delta_low_count"] = delta["post_n_low"] - delta["pre_n_low"]
    sns.boxplot(data=delta, x="model", y="delta_low_count", ax=axes[1], color="#72B7B2")
    axes[1].axhline(0, color="black", lw=1)
    axes[1].tick_params(axis="x", rotation=25)
    axes[1].set_title("Change in lower-edge outliers (post - pre)")
    return fig


if RUN_MONTE_CARLO:
    monte_carlo_results = run_monte_carlo(CONFIG)
    monte_carlo_summary = summarize_monte_carlo(monte_carlo_results)
    monte_carlo_results.to_csv(OUTPUT_DIR / "monte_carlo_results.csv", index=False)
    monte_carlo_summary.to_csv(OUTPUT_DIR / "monte_carlo_summary.csv", index=False)
    display(monte_carlo_summary)
    figure = plot_monte_carlo(monte_carlo_results)
    figure.savefig(OUTPUT_DIR / "monte_carlo_lower_outliers.png", dpi=170)
    plt.show()
else:
    print("Monte CarloはRUN_MONTE_CARLO=Trueで実行します。")


## 7. 下側spike強度のsweep

理論境界 `tau = 1 - sqrt(gamma)` の前後で、下側外れ値の発生とDEによる回復がどう変わるかを見る。計算量を抑えるためAだけを対象とする。


In [ ]:
from dataclasses import replace


def run_tau_sweep(config, tau_grid):
    records = []
    for tau in tau_grid:
        local_config = replace(config, tau=float(tau))
        for repeat in range(config.repeats):
            seed = config.base_seed + int(round(tau * 10_000)) + repeat
            row, _ = analyze_one("A_separable_lower_spike", local_config, seed)
            records.append(row)
    return pd.DataFrame(records)


if RUN_TAU_SWEEP:
    tau_results = run_tau_sweep(CONFIG, TAU_GRID)
    tau_results.to_csv(OUTPUT_DIR / "tau_sweep_results.csv", index=False)
    tau_long = tau_results.melt(
        id_vars=["tau", "seed"], value_vars=["pre_n_low", "post_n_low"],
        var_name="condition", value_name="lower_outlier_count",
    )
    tau_long["condition"] = tau_long["condition"].map({"pre_n_low": "Before DE", "post_n_low": "After DE"})
    figure, axis = plt.subplots(figsize=(8, 4.5), layout="constrained")
    sns.lineplot(
        data=tau_long, x="tau", y="lower_outlier_count", hue="condition",
        estimator="median", errorbar=("pi", 80), marker="o", ax=axis,
    )
    axis.axvline(1 - np.sqrt(CONFIG.p / CONFIG.n), color="black", ls="--", label="theory boundary")
    axis.set_title("Lower outliers across the population-spike transition")
    axis.set_ylabel("lower outlier count")
    axis.legend()
    figure.savefig(OUTPUT_DIR / "tau_sweep.png", dpi=170)
    plt.show()
else:
    print("tau sweepはRUN_TAU_SWEEP=Trueで実行します。")


## 8. 実行後の判定

理想的な結果は、次を同時に満たすこと。

1. Aで `pre_n_low > 0`, `post_n_low` がnull相当まで減る。
2. Aで `pre_n_high` がほぼ0で、下側だけのモデル化になっている。
3. Bで上側信号を残しつつ、下側だけが改善する。
4. C・DではAよりもDE後の `n_low` または `fullKS` が大きく残る。
5. A・Bで真の行分散と推定スケールの `x_log_corr` が高い。

有限標本では仮定内でも下側外れ値が毎回0になるとは限らない。単一seedではなく、Monte Carloの中央値・10–90%区間で判断する。


In [ ]:
manifest = {
    "config": asdict(CONFIG),
    "models": MODEL_NAMES,
    "tau_grid": TAU_GRID,
    "spectrum_scale": "lambda = singular_value**2 / n",
    "pre_reference": "MP with BEMA variance",
    "post_reference": "MP with variance 1",
    "de_function": "funcs1.dyson_equalizer_algorithm1",
}
manifest_text = json.dumps(manifest, ensure_ascii=False, sort_keys=True, default=list)
manifest["config_sha256"] = hashlib.sha256(manifest_text.encode()).hexdigest()
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2, default=list))
print("manifest:", OUTPUT_DIR / "manifest.json")
